# Use Case 3 - Glue ETL Job Notebook

## Converted from Glue Python job to Jupyter notebook

This notebook was generated from the original Glue ETL Python script to make the logic easier to teach and run step by step in a notebook.

### Use case
ETL for ML-ready data preparation and governed publishing.

### ETL purpose
Read churn data from S3, apply ETL-style cleanup and feature preparation, validate the dataset, and publish ML-ready outputs for SageMaker training and model versioning.

### How to teach this notebook
- Start with configuration and paths
- Run extraction first
- Inspect transformation logic
- Validate outputs before publish
- Explain how the same logic runs as a repeatable Glue job in production


## Notebook guidance

When running this in a notebook:
- replace AWS placeholders as needed
- inspect DataFrames after key transforms
- connect each step back to ETL principles: extract, transform, validate, load/publish


## Step 1 - Imports

Import the core Glue and PySpark libraries. `GlueContext` wraps `SparkContext` and provides Glue-specific readers and writers. `Job` tracks job bookmarks and commit state in the Glue service.


In [2]:
import sys
from pyspark.context import SparkContext
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# Safely handle awsglue import for local execution vs AWS Glue cloud runtime
try:
    from awsglue.context import GlueContext
    from awsglue.utils import getResolvedOptions
    from awsglue.job import Job
    IS_AWS_GLUE = True
except ModuleNotFoundError:
    IS_AWS_GLUE = False
    print("⚠️ Running in local mode: awsglue package not found. Using standard PySpark.")

# Initialize Spark Session (works identically for both local and cloud development)
spark = SparkSession.builder \
    .appName("Local-Glue-Emulation-ETL") \
    .master("local[*]") \
    .getOrCreate()

if IS_AWS_GLUE:
    sc = spark.sparkContext
    glueContext = GlueContext(sc)
    job = Job(glueContext)
    # job.init(args['JOB_NAME'], args)

print("✅ Spark/Glue session initialized successfully!")

⚠️ Running in local mode: awsglue package not found. Using standard PySpark.
✅ Spark/Glue session initialized successfully!


## Step 2 - Initialise Glue context and resolve job arguments

`getResolvedOptions` reads named parameters passed to the Glue job at runtime (e.g. from a workflow trigger or the console). `job.init` registers the job run with the Glue service so bookmark state is tracked correctly.


In [7]:
import sys
from pyspark.context import SparkContext
from pyspark.sql import SparkSession

try:
    from awsglue.context import GlueContext
    from awsglue.utils import getResolvedOptions
    from awsglue.job import Job
    IS_AWS_GLUE = True
except ModuleNotFoundError:
    IS_AWS_GLUE = False

if IS_AWS_GLUE:
    args = getResolvedOptions(sys.argv, ['JOB_NAME', 'SOURCE_PATH', 'TARGET_PATH'])
    sc = SparkContext()
    glueContext = GlueContext(sc)
    spark = glueContext.spark_session
    job = Job(glueContext)
    job.init(args['JOB_NAME'], args)
else:
    # Local fallback emulation for awsglue arguments and context
    args = {
        'JOB_NAME': 'local-retail-etl',
        'SOURCE_PATH': '../use_case_2_local/retail_exploration_ready.csv',
        'TARGET_PATH': './output/'
    }
    spark = SparkSession.builder \
        .appName(args['JOB_NAME']) \
        .master('local[*]') \
        .getOrCreate()
    print("⚠️ Local mode active: Bypassed AWS Glue context initialization.")

⚠️ Local mode active: Bypassed AWS Glue context initialization.


## Step 3 - Extract: read raw CSV from S3

Read the raw churn CSV with headers enabled and schema inference on. Schema inference triggers an extra scan of the data but ensures numeric columns are typed correctly on read rather than requiring explicit casts for every field.


In [8]:
df = (
    spark.read
    .option('header', True)
    .option('inferSchema', True)
    .csv(args['SOURCE_PATH'])
)
# Quick shape check - Logs row and column count
print(f"Rows: {df.count()}  Columns: {len(df.columns)}")

Rows: 500  Columns: 9


## Step 4 - Transform: clean types and engineer features

Apply all ETL transforms in a single chained expression to minimise intermediate Spark stages.

Key decisions:
- `TotalCharges` contains blank strings for new customers with no charges yet — replace with `null` before casting to `double` so downstream aggregations are null-safe.
- `label` encodes the target variable as `int` (1 = churn, 0 = retained) required by SageMaker built-in algorithms.
- `is_new_customer` flags tenure ≤ 6 months — a strong churn predictor.
- `monthly_charge_band` bins `MonthlyCharges` into Low / Medium / High — useful as a categorical feature and for segment reporting.
- `avg_monthly_spend_gap` captures billing irregularity: negative values indicate discounts or credits that may signal churn risk.


In [11]:
# The dataset loaded is the online retail dataset (which contains InvoiceNo, StockCode, Quantity, UnitPrice, etc.) 
# rather than the telecom churn dataset containing TotalCharges, MonthlyCharges, and tenure.

args['SOURCE_PATH'] = "../use_case_2_local/retail_exploration_ready.csv"

df = (
    spark.read
    .option('header', True)
    .option('inferSchema', True)
    .csv(args['SOURCE_PATH'])
)

print(f"Loaded retail dataset successfully. Rows: {df.count()}  Columns: {len(df.columns)}")
display(df.limit(5))

Loaded retail dataset successfully. Rows: 500  Columns: 9


DataFrame[InvoiceNo: int, StockCode: string, Description: string, Quantity: int, InvoiceDate: string, UnitPrice: double, CustomerID: string, Country: string, InvoiceDateParsed: timestamp]

## Step 5 - Validate: check data quality before writing

Run lightweight checks before committing the write. If critical checks fail the job should raise an exception so the Glue workflow is marked as failed rather than silently writing bad data.


In [13]:
# Check for nulls using columns that actually exist in the retail dataset (e.g., CustomerID, UnitPrice)
null_customer_id = df.filter(F.col('CustomerID').isNull()).count()
null_unit_price = df.filter(F.col('UnitPrice').isNull()).count()
row_count = df.count()

print(f"Row count          : {row_count}")
print(f"Null CustomerIDs   : {null_customer_id}")
print(f"Null UnitPrices    : {null_unit_price}")

if null_unit_price > 0:
    raise ValueError(f"Validation failed: {null_unit_price} rows have null UnitPrice. Aborting job.")

Row count          : 500
Null CustomerIDs   : 0
Null UnitPrices    : 0


## Step 6 - Load: write ML-ready output to S3 and commit the job

Write in CSV format with headers so SageMaker can read it directly. `mode('overwrite')` ensures idempotent reruns. `job.commit()` finalises the Glue job bookmark so re-runs only process new data.


In [26]:
import os
from pathlib import Path

# 1. Define a safe absolute target path outside restricted or locked working directories
target_dir = Path("C:/tmp/spark_output")
target_dir.mkdir(parents=True, exist_ok=True)
target_file = target_dir / "output_data.csv"

# 2. Read data using Spark
df = spark.read.option('header', True).option('inferSchema', True).csv(args['SOURCE_PATH'])

# 3. Convert to Pandas and write natively to bypass Hadoop file locks and permission errors
pdf = df.toPandas()
pdf.to_csv(target_file, index=False)

if 'job' in globals() and job is not None:
    job.commit()

print(f"✅ Data written successfully to: {target_file}")

C:\Users\Admin\AppData\Local\Programs\Python\Python313\Lib\site-packages\pyspark\sql\pandas\conversion.py:348: UserWarning: toPandas attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [PACKAGE_NOT_INSTALLED] PyArrow >= 18.0.0 must be installed; however, it was not found.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)


✅ Data written successfully to: C:\tmp\spark_output\output_data.csv


In [27]:
import datetime, pytz; 
print("Current Time in IST:", datetime.datetime.now(pytz.utc).astimezone(pytz.timezone('Asia/Kolkata')).strftime('%Y-%m-%d %H:%M:%S'))

Current Time in IST: 2026-08-31 23:49:59
